In [0]:

from pyspark.sql import functions as F

df_transaction = spark.table(
    "hackathon.shared_datasets.train_transaction"
)

df_identity = spark.table(
    "hackathon.shared_datasets.train_identity"
)

df_narratives = spark.table(
    "hackathon.shared_datasets.synthetic_crypto_fraud_narratives"
)




In [0]:

datasets = {
    "Transactions": df_transaction.count(),
    "Identity": df_identity.count(),
    "Narratives": df_narratives.count()
}

datasets


{'Transactions': 590540, 'Identity': 144233, 'Narratives': 1000}

In [0]:

df_joined = (
    df_transaction
    .join(
        df_identity,
        on="TransactionID",
        how="left"
    )
)


In [0]:

print(f"Rows: {df_joined.count():,}")
print(f"Columns: {len(df_joined.columns)}")


Rows: 590,540
Columns: 434


In [0]:

label_dist = (
    df_joined
    .groupBy("isFraud")
    .count()
    .orderBy("isFraud")
)

display(label_dist)


isFraud,count
0,569877
1,20663


In [0]:

fraud_rate = (
    df_joined
    .agg(
        F.mean("isFraud").alias("fraud_rate")
    )
)

display(fraud_rate)


fraud_rate
0.03499000914417313


In [0]:

total_rows = df_joined.count()

null_counts = df_joined.select(
    [
        F.sum(
            F.when(F.col(c).isNull(), 1)
            .otherwise(0)
        ).alias(c)
        for c in df_joined.columns
    ]
)


In [0]:

import pandas as pd

null_row = null_counts.collect()[0]

null_df = pd.DataFrame({
    "column": df_joined.columns,
    "null_count": [null_row[c] for c in df_joined.columns]
})

null_df["null_pct"] = (
    null_df["null_count"] /
    total_rows * 100
)

null_df.sort_values(
    "null_pct",
    ascending=False
).head(20)


,column,null_count,null_pct
417,id_24,585793,99.196159
418,id_25,585408,99.130965
400,id_07,585385,99.127070
401,id_08,585385,99.127070
414,id_21,585381,99.126393
419,id_26,585377,99.125715
415,id_22,585371,99.124699
420,id_27,585371,99.124699
416,id_23,585371,99.124699
14,dist2,552913,93.628374
